# Colab backend pipeline

Notebook này chỉ chạy trên Google Colab.
- Upload project ZIP hoặc mount code trên Drive
- Upload video vào Colab hoặc Drive
- Chạy backend render video trên Colab
- Copy kết quả về Drive để frontend local đọc

> Bạn chỉ cần đưa video vào. Backend Colab lo phần còn lại.


In [ ]:
# Upload project ZIP và video file vào Colab
from pathlib import Path
from google.colab import files
import zipfile
import shutil

upload_root = Path('/content/uploaded')
project_root = upload_root / 'project'
video_root = upload_root / 'video'
project_root.mkdir(parents=True, exist_ok=True)
video_root.mkdir(parents=True, exist_ok=True)

print('Upload project ZIP (nếu bạn chưa mount code trong Drive) và/hoặc video file.')
print('Project sẽ được giải nén tại', project_root)
print('Video sẽ được lưu tại', video_root)

uploaded = files.upload()
for filename, data in uploaded.items():
    dest = upload_root / filename
    with open(dest, 'wb') as f:
        f.write(data)
    print('Saved:', dest)
    if filename.lower().endswith('.zip'):
        with zipfile.ZipFile(dest, 'r') as z:
            z.extractall(project_root)
        print('Extracted project ZIP to', project_root)
    elif filename.lower().endswith(('.mp4', '.mov', '.mkv', '.webm')):
        video_dest = video_root / filename
        shutil.copy2(dest, video_dest)
        print('Saved video to', video_dest)

print('Upload completed.')
print('Project files in:', project_root)
print('Video files in:', video_root)


In [ ]:
# Mount Google Drive and install dependencies
from pathlib import Path
import subprocess
import sys
import os

requirements_path = None
project_root = None

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('Drive mount skipped:', exc)

SEARCH_PATHS = [
    Path('/content/uploaded/project'),
    Path('/content/drive/MyDrive'),
    Path('/content/drive/My Drive'),
]

search_roots = [Path.cwd()]
search_roots.extend(SEARCH_PATHS)

print('search_roots:', [str(root) for root in search_roots])
for root in search_roots:
    if not root.exists():
        print('  missing:', root)
        continue
    for candidate in root.rglob('requirements.txt'):
        candidate_parent = candidate.parent
        if (candidate_parent / 'app').exists() or (candidate_parent / 'config_pipeline_full.yaml').exists() or (candidate_parent / 'config_pipeline.yaml').exists():
            requirements_path = candidate
            project_root = candidate_parent
            print('Found project root at', project_root)
            break
    if project_root:
        break

if not requirements_path:
    raise FileNotFoundError('Project not found. Upload project ZIP into Colab or mount project on Drive.')

print('Installing requirements from', requirements_path)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp', 'pyyaml'], check=True)


Installing requirements from C:\Users\Administrator\Desktop\3\requirements.txt


CompletedProcess(args=['python', '-m', 'pip', 'install', '-q', 'yt-dlp', 'pyyaml'], returncode=0)

In [ ]:
# Set repository root on Colab
from pathlib import Path
import os
import sys

REPO_DIR = project_root if 'project_root' in globals() and project_root is not None else Path('/content/uploaded/project')
if not REPO_DIR.exists() or not (REPO_DIR / 'app').exists():
    for candidate in [Path('/content/drive/MyDrive'), Path('/content/uploaded/project')]:
        if candidate.exists() and (candidate / 'app').exists():
            REPO_DIR = candidate
            break

if not REPO_DIR.exists() or not (REPO_DIR / 'app').exists():
    raise FileNotFoundError('Project repo not found. Upload project ZIP to Colab or mount project on Drive.')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
(REPO_DIR / 'outputs').mkdir(parents=True, exist_ok=True)
print('Using project root:', REPO_DIR)
print('cwd:', Path.cwd())
print('sys.path[0]:', sys.path[0])


Drive mount skipped: No module named 'google'
No GitHub repo URL provided; using the detected workspace directory.
Using project root: C:\Users\Administrator\Desktop\3
cwd: C:\Users\Administrator\Desktop\3
sys.path[0]: C:\Users\Administrator\Desktop\3


In [ ]:
# Check ffmpeg availability on Colab
from shutil import which
ffmpeg_path = which('ffmpeg')
print('ffmpeg path:', ffmpeg_path)


ffmpeg not found in PATH and no local portable version was detected.
Checked candidates:
  - C:\content\video-pipeline\ffmpeg_bin\ffmpeg-master-latest-win64-gpl\bin\ffmpeg.exe False
  - C:\ffmpeg\bin\ffmpeg.exe False
  - C:\Program Files\ffmpeg\bin\ffmpeg.exe False
  - C:\Program Files (x86)\ffmpeg\bin\ffmpeg.exe False
  - /usr/bin/ffmpeg False
  - /usr/local/bin/ffmpeg False


In [ ]:
# Run backend pipeline and copy result to Drive
import sys
import shutil
from pathlib import Path

sys.path.insert(0, str(REPO_DIR))
from app.plugins.runner import run_from_config

video_candidates = [
    Path('/content/drive/MyDrive/video.mp4'),
    Path('/content/drive/MyDrive/video.mov'),
    Path('/content/uploaded/video/video.mp4'),
    Path('/content/uploaded/video/video.mov'),
]
video_input_path = next((p for p in video_candidates if p.exists()), None)

if video_input_path is None:
    uploaded_dir = Path('/content/uploaded/video')
    uploaded_files = sorted([p for p in uploaded_dir.glob('*') if p.is_file()])
    video_input_path = uploaded_files[0] if uploaded_files else None

if video_input_path is None or not video_input_path.exists():
    raise FileNotFoundError(
        'Video file not found. Upload video via the upload cell or place it in /content/drive/MyDrive/video.mp4.'
    )

print('Using video input:', video_input_path)
result = run_from_config('config_pipeline_full.yaml', video_url=str(video_input_path))
print(result)

output_local = REPO_DIR / 'outputs' / 'final.mp4'
drive_output = Path('/content/drive/MyDrive/colab_outputs/final.mp4')
drive_output.parent.mkdir(parents=True, exist_ok=True)
if output_local.exists():
    shutil.copy2(output_local, drive_output)
    print('Copied final video to', drive_output)
else:
    print('Rendered output not found at', output_local)


[Pipeline] start -> Download Video
[Pipeline] download ok -> success
[Pipeline] end -> Download Video
[Pipeline] start -> ASR
[Pipeline] asr ok -> Whisper placeholder
[Pipeline] end -> ASR
[Pipeline] start -> OCR
[Pipeline] end -> OCR
[Pipeline] start -> AI Translate
[Pipeline] end -> AI Translate
[Pipeline] start -> Translation
[TranslationStep] provider=openrouter model=google/gemini-2.5-flash-lite api_key_present=True
[Pipeline] translate ok -> Chỗ giữ chỗ thì thầm
[Pipeline] end -> Translation
[Pipeline] start -> TTS
[Pipeline] end -> TTS
[Pipeline] start -> FFmpeg Render
[Pipeline] render ok -> success
[Pipeline] end -> FFmpeg Render
{'url': None, 'output_dir': 'outputs', 'video_path': 'C:\\Users\\Administrator\\Desktop\\3\\downloads\\raw.mp4.mp4', 'input_path': 'C:\\Users\\Administrator\\Desktop\\3\\downloads\\raw.mp4.mp4', 'timeout': 120, 'download_status': 'success', 'download_source': 'local_file', 'engine': 'edge-tts', 'language': 'ch', 'model': 'google/gemini-2.5-flash-lite'